In [ ]:
# ============================================
# MLflow Manual Logging - Linear Regression
# Dataset: California Housing (sklearn built-in)
# ============================================

import logging
import warnings
import numpy as np
import pandas as pd
from urllib.parse import urlparse

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

logging.basicConfig(level=logging.WARN)
logger = logging.getLogger(__name__)
warnings.filterwarnings("ignore")
np.random.seed(40)

# --- Helper function ---
def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2

# --- Load California Housing dataset ---
housing = fetch_california_housing(as_frame=True)
data = housing.frame
print(f"Dataset shape: {data.shape}")
print(data.head())

# Split the data into training and test sets (0.75, 0.25)
train, test = train_test_split(data)

# Target column is "MedHouseVal" (median house value)
train_x = train.drop(["MedHouseVal"], axis=1)
test_x = test.drop(["MedHouseVal"], axis=1)
train_y = train[["MedHouseVal"]]
test_y = test[["MedHouseVal"]]

alpha = 0.5
l1_ratio = 0.5

# --- MLflow Run ---
with mlflow.start_run():
    lr = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42)
    lr.fit(train_x, train_y)

    predicted = lr.predict(test_x)

    (rmse, mae, r2) = eval_metrics(test_y, predicted)

    print(f"ElasticNet model (alpha={alpha:.4f}, l1_ratio={l1_ratio:.4f}):")
    print(f"  RMSE: {rmse}")
    print(f"  MAE: {mae}")
    print(f"  R2: {r2}")

    # Log params and metrics
    mlflow.log_param("alpha", alpha)
    mlflow.log_param("l1_ratio", l1_ratio)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("mae", mae)

    # Log model
    predictions = lr.predict(train_x)
    signature = infer_signature(train_x, predictions)

    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

    if tracking_url_type_store != "file":
        mlflow.sklearn.log_model(
            lr, "model", registered_model_name="ElasticnetHousingModel", signature=signature
        )
    else:
        mlflow.sklearn.log_model(lr, "model", signature=signature)

print("\nModel logged successfully!")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")